In [9]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib 
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

In [10]:
#Load the data

raw = pd.read_csv('superstore_sales (1).csv')

In [11]:
#Data preprocessing

returned_ids = raw.loc[raw['Row ID'] == 'Yes', 'Order ID'].unique()
df = raw[raw['Order Date'].notna()].copy()
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Year'], df['Month'] = df['Order Date'].dt.year, df['Order Date'].dt.month
df['Returned'] = df['Order ID'].isin(returned_ids)

In [12]:
#Purchase volume

cust = df.groupby('Customer ID').agg(Sales=('Sales', 'sum'), Profit=('Profit', 'sum'),
                                     Orders=('Order ID', 'nunique')).sort_values('Sales', ascending=False)
cust['CumShare'] = cust.Sales.cumsum() / cust.Sales.sum()
cust['Pct'] = np.arange(1, len(cust) + 1) / len(cust)
cust['Tier'] = pd.qcut(cust.Sales.rank(method='first'), 4, labels=['Bottom 25%', '25-50%', '50-75%', 'Top 25%'])
tier = cust.groupby('Tier', observed=True)[['Sales', 'Profit']].sum()

In [27]:
#Graphs

fig, ax = plt.subplots(1, 3, figsize=(22, 6.5))
sns.histplot(cust.Orders, bins=range(1, 19), ax=ax[0], color='#4C72B0')
ax[0].set(title='Orders per customer', xlabel='Orders (4 yrs)', ylabel='Customers')
ax[1].plot(cust.Pct, cust.CumShare, lw=3, color='#DD8452'); ax[1].plot([0, 1], [0, 1], '--', color='grey')
ax[1].xaxis.set_major_formatter('Pct'); ax[1].yaxis.set_major_formatter('Pct')
ax[1].set(title='Revenue concentration (Lorenz)', xlabel='Share of customers', ylabel='Share of revenue')
ax[1].axvline(.2, ls=':', color='k'); ax[1].annotate(f"Top 20% = {cust.CumShare[cust.Pct <= .2].iloc[-1]:.0%}", (.22, .35))
t = tier.reset_index().melt('Tier'); sns.barplot(t, x='Tier', y='value', hue='variable', ax=ax[2])
ax[2].yaxis.set_major_formatter('k'); ax[2].set(title='Sales & profit by customer quartile', xlabel='', ylabel='')
ax[2].legend(title='')
plt.tight_layout(); plt.savefig('1_customer_volume.png', dpi=130); plt.close()

In [21]:
#Monthly Seasonality

mo = df.groupby(['Year', 'Month']).Sales.sum().reset_index()
avg = df.groupby('Month').Sales.sum(); idx = avg / avg.mean()
fig, ax = plt.subplots(1, 2, figsize=(22, 7))
sns.lineplot(mo, x='Month', y='Sales', hue='Year', palette='viridis', marker='o', lw=3, ax=ax[0])
ax[0].yaxis.set_major_formatter('k'); ax[0].set_xticks(range(1, 13)); ax[0].set(title='Monthly sales by year')
cols = ['#C44E52' if v >= 1.4 else '#4C72B0' if v >= .7 else '#8C8C8C' for v in idx]
ax[1].bar(idx.index, idx.values, color=cols); ax[1].axhline(1, color='k', ls='--')
ax[1].set_xticks(range(1, 13)); ax[1].set(title='Seasonality index (1.0 = average month)', xlabel='Month')
plt.tight_layout(); plt.savefig('2_monthly_seasonality.png', dpi=130); plt.close()

In [25]:
#Revenue by Category

cat = df.groupby('Category').agg(Sales=('Sales', 'sum'), Profit=('Profit', 'sum'))
cat['Margin'] = cat.Profit / cat.Sales
sub = df.groupby(['Category', 'Sub-Category']).agg(Sales=('Sales', 'sum'), Profit=('Profit', 'sum')).reset_index()
sub['Margin'] = sub.Profit / sub.Sales; sub = sub.sort_values('Profit')
fig, ax = plt.subplots(1, 3, figsize=(24, 8))
cy = df.groupby(['Year', 'Category']).Sales.sum().unstack()
cy.plot(kind='bar', stacked=True, ax=ax[0], rot=0); ax[0].yaxis.set_major_formatter('k')
ax[0].set(title='Sales by category & year', xlabel=''); ax[0].legend(title='')
c = cat.reset_index().melt('Category', ['Sales', 'Profit'])
sns.barplot(c, x='Category', y='value', hue='variable', ax=ax[1]); ax[1].yaxis.set_major_formatter('k')
for i, (n, r) in enumerate(cat.iterrows()):
    ax[1].text(i, r.Sales * 1.02, f'{r.Margin:.1%} margin', ha='center', fontsize=13)
ax[1].set(title='Sales vs profit', xlabel='', ylabel=''); ax[1].legend(title='')
ax[2].barh(sub['Sub-Category'], sub.Profit, color=np.where(sub.Profit < 0, '#C44E52', '#55A868'))
ax[2].xaxis.set_major_formatter('k'); ax[2].set(title='Profit by sub-category')
plt.tight_layout(); plt.savefig('3_category_revenue.png', dpi=130); plt.close()

In [29]:
#Discount

labels = ['0%', '1-10%', '11-20%', '21-30%', '31-50%', '51-70%', '71-80%']
df['Band'] = pd.cut(df.Discount, [-.001, 0, .1, .2, .3, .5, .7, .8], labels=labels)
d = df.groupby('Band', observed=True).agg(Lines=('Sales', 'size'), Sales=('Sales', 'sum'), Profit=('Profit', 'sum'),
                                          Qty=('Quantity', 'mean'), Loss=('Profit', lambda s: (s < 0).mean()))
d['Margin'] = d.Profit / d.Sales
fig, ax = plt.subplots(1, 3, figsize=(24, 7))
ax[0].bar(d.index, d.Margin, color=np.where(d.Margin < 0, '#C44E52', '#55A868')); ax[0].axhline(0, color='k')
ax[0].yaxis.set_major_formatter('Pct'); ax[0].set(title='Profit margin by discount band', xlabel='Discount')
ax[1].bar(d.index, d.Qty, color='#4C72B0'); ax[1].set(title='Avg units per line (flat = no volume lift)', xlabel='Discount', ylim=(0, 5))
hm = df.pivot_table(index='Category', columns='Band', values='Profit', aggfunc='sum', observed=True)
sales = df.pivot_table(index='Category', columns='Band', values='Sales', aggfunc='sum', observed=True)
sns.heatmap(hm / sales, annot=True, fmt='.0%', cmap='RdYlGn', center=0, ax=ax[2], cbar=False)
ax[2].set(title='Margin: category x discount', xlabel='Discount', ylabel='')
plt.tight_layout(); plt.savefig('4_discount_sensitivity.png', dpi=130); plt.close()

In [31]:
#Summary

with pd.ExcelWriter('superstore_analysis_tables.xlsx') as w:
    cust.head(25).drop(columns=['Pct']).to_excel(w, sheet_name='Top25_Customers')
    tier.to_excel(w, sheet_name='Customer_Tiers')
    df.groupby(['Year', 'Month']).Sales.sum().unstack(0).to_excel(w, sheet_name='Monthly_Sales')
    idx.rename('Seasonality_Index').to_frame().to_excel(w, sheet_name='Seasonality')
    cat.to_excel(w, sheet_name='Category'); sub.to_excel(w, sheet_name='SubCategory', index=False)
    d.to_excel(w, sheet_name='Discount_Bands')
print('done')

done
